# SANOS — Option Price LP (no IV roundtrip)

Fits SANOS directly on **dollar bid/ask prices** — no implied vol conversion.

**Calls and puts are fitted by independent LP solves per expiry.**

Model: $\hat{C}(K) = \sum_j q_j \cdot \text{BS}_\text{call}(K_j^\text{anchor}, K, \eta V_\text{atm})$  
Put model: $\hat{P}(K) = \hat{C}(K) - e^{-rT}(F - K) = A_\text{put} \cdot q$

$q$ is a probability vector (non-negative, unit sum, martingale) → strictly arbitrage-free.  
LP bounds come directly from normalized bid/ask prices; the only IV inversion is one scalar Brentq at the ATM mid to set $V_\text{atm}$.

**Caveat:** Batch fit — all day's quotes feed the LP simultaneously.

In [ ]:
import importlib as imp
import os
import sys
from datetime import date

import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import plotly.subplots as psp
import polars as pl

project_root = os.path.abspath("/Users/elie/Documents/GitHub/quant_data/")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

parent_dir = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

sanos_dir = os.path.abspath(os.path.dirname("__file__"))
if sanos_dir not in sys.path:
    sys.path.insert(0, sanos_dir)

shared_functions_dir = os.path.join(project_root, "personal", "shared_functions")
if shared_functions_dir not in sys.path:
    sys.path.insert(0, shared_functions_dir)

os.environ["QUANTDATA_RAW_DATA"] = "!/quantdata/raw"
os.environ["QUANTDATA_TRAINING_DATA"] = "!/quantdata/training"
os.environ["QUANTDATA_PROCESSING_DATA"] = "!/quantdata/processing"
os.environ["QUANTDATA_CACHE_MODE"] = "on"
os.environ["QUANTDATA_TEST_MODE"] = "0"

import quantdata as _

imp.reload(_)
import quantdata.biz_calendar as _

imp.reload(_)
import quantdata.massive.client as _

imp.reload(_)
import quantdata.massive.options.contracts as _

imp.reload(_)

import sanos_option

import personal.shared_functions.utils as utils
from quantdata.massive.client import MassiveClient
from quantdata.massive.taq import TAQMatchConfig

## Config — change these to experiment

In [ ]:
imp.reload(sanos_option)

cfg = sanos_option.SanosOptionConfig(
    N=30,
    eta=0.85,  # ≥0.8 for short-dated SPY (T≈0.008 yr, small V_atm → need wide basis)
    k_min=-0.25,
    k_max=0.15,
    solver="CLARABEL",
    otm_calls_only=True,  # LP uses k≥0 calls only
    otm_puts_only=True,  # LP uses k<0 puts only
)
print(cfg)

## Data loading

In [3]:
df_tickers = pl.read_csv("../../spy_options_tickers.csv")

In [4]:
trade_date = date(2024, 1, 3)
client = MassiveClient()
df_spy = client.fetch_trades("SPY", trade_date)

## Build price observation stream

`build_option_observations` fetches quotes + trades, estimates the implied forward via the 2×2 put-call parity system, and tags each row with `T`, `forward`, `discount_factor`, and `log_moneyness`.  
Output columns: `ts_ns, strike, is_call, log_moneyness, T, observed_price, bid_price, ask_price, forward, discount_factor`.

In [ ]:
imp.reload(sanos_option)

df_quotes, df_trades = sanos_option.build_option_observations(
    df_tickers=df_tickers,
    trade_date=trade_date,
    df_spy=df_spy,
    min_trades=100,
    simple=False,  # use full 2×2 PCP system for forward
    cfg=cfg,
)
print(
    f"Quotes: {df_quotes.shape[0]:,} rows  |  {df_quotes['strike'].n_unique()} strikes  |  {df_quotes['is_call'].sum()} calls  {(~df_quotes['is_call']).sum()} puts"
)
print(f"Trades: {df_trades.shape[0]:,} rows")
df_quotes.head()

## Fit SANOS in price space

One LP per (expiry, type): calls fitted with `A_call`, puts with `A_put = A_call − exp(anchor) + exp(k)`.  
Calls and puts run in parallel via `ThreadPoolExecutor`.

In [ ]:
import time

t0 = time.perf_counter()
df_q_sanos, df_t_sanos = sanos_option.run_sanos_option_batch(df_quotes[::10], df_trades[::10], cfg)
elapsed = time.perf_counter() - t0
print(f"\nTotal fit + evaluation: {elapsed:.2f}s")

## Quick residual summary

In [ ]:
df_t_clean = df_t_sanos.filter(
    pl.col("sanos_matched").is_not_null()
    & pl.col("sanos_matched").is_not_nan()
    & pl.col("trade_residual").is_not_null()
    & pl.col("trade_residual").is_not_nan()
)
rmse = float((df_t_clean["trade_residual"] ** 2).mean() ** 0.5)
bias = float(df_t_clean["trade_residual"].mean())
mae = float(df_t_clean["trade_residual"].abs().mean())
print(f"SANOS trade RMSE  (price $) : {rmse:.5f}")
print(f"SANOS trade bias  (price $) : {bias:.5f}")
print(f"SANOS trade MAE   (price $) : {mae:.5f}")
print(f"Evaluated trades            : {df_t_clean.shape[0]:,} / {df_t_sanos.shape[0]:,}")

# Per-type breakdown
for is_call, label in [(True, "calls"), (False, "puts")]:
    sub = df_t_clean.filter(pl.col("is_call") == is_call)
    if sub.is_empty():
        continue
    r = float((sub["trade_residual"] ** 2).mean() ** 0.5)
    b = float(sub["trade_residual"].mean())
    print(f"  {label}: RMSE={r:.5f}  bias={b:.5f}  n={sub.shape[0]:,}")

## Trade residuals vs log-moneyness

In [ ]:
fig = psp.make_subplots(rows=1, cols=2, subplot_titles=["Calls", "Puts"], shared_yaxes=True)

for col_idx, (is_call, label, color) in enumerate(
    [(True, "call", "royalblue"), (False, "put", "firebrick")]
):
    sub = df_t_clean.filter(pl.col("is_call") == is_call)
    if sub.is_empty():
        continue
    fig.add_trace(
        go.Scatter(
            x=sub["log_moneyness"].to_list(),
            y=sub["trade_residual"].to_list(),
            mode="markers",
            name=label,
            marker=dict(color=color, size=4, opacity=0.4),
            showlegend=False,
        ),
        row=1,
        col=col_idx + 1,
    )
    fig.add_hline(y=0, line=dict(color="black", dash="dash", width=1), row=1, col=col_idx + 1)

fig.update_xaxes(title_text="log-moneyness")
fig.update_yaxes(title_text="residual (trade − SANOS, $)", col=1)
fig.update_layout(height=400, title_text="Trade residuals in price space  (trade − sanos_matched)")

## Fitted price smile — calls and puts per expiry

Each row is one expiry; **calls on the left, puts on the right**.  
Shaded band = bid/ask spread (median across day). Line = SANOS at market strikes. Crosses = market mid.

In [ ]:
T_r_expr = (pl.col("T") * 200).round(0) / 200  # 0.005 yr bins — one per expiry

# Median across day per (expiry, strike, type)
snap = (
    df_q_sanos.filter(pl.col("sanos_price").is_not_null() & pl.col("sanos_price").is_not_nan())
    .with_columns(T_r_expr.alias("T_round"))
    .group_by(["T_round", "strike", "is_call"])
    .agg(
        pl.col("T").first(),
        pl.col("log_moneyness").median().alias("log_moneyness"),
        pl.col("sanos_price").median().alias("sanos_price_med"),
        pl.col("observed_price").median().alias("mid_price_med"),
        pl.col("bid_price").median().alias("bid_price_med"),
        pl.col("ask_price").median().alias("ask_price_med"),
    )
    .sort(["T_round", "is_call", "log_moneyness"])
)

expiries = sorted(snap["T_round"].unique().to_list())
n_exp = len(expiries)

fig = psp.make_subplots(
    rows=n_exp,
    cols=2,
    column_titles=["Calls (k ≥ 0)", "Puts (k < 0)"],
    subplot_titles=[
        title
        for T_r in expiries
        for title in [
            f"T = {float(snap.filter(pl.col('T_round') == T_r)['T'].first()):.4f} yr",
            "",
        ]
    ],
    horizontal_spacing=0.08,
    vertical_spacing=0.14 if n_exp > 1 else 0.10,
    shared_xaxes=False,
)

for i, T_r in enumerate(expiries):
    row = i + 1
    show = i == 0

    for col_idx, (is_call, color, fill_color) in enumerate(
        [
            (True, "royalblue", "rgba(65,105,225,0.18)"),
            (False, "firebrick", "rgba(178,34,34,0.18)"),
        ]
    ):
        col = col_idx + 1
        s = snap.filter((pl.col("T_round") == T_r) & (pl.col("is_call") == is_call)).sort(
            "log_moneyness"
        )
        if s.is_empty():
            continue

        ks = s["log_moneyness"].to_list()
        bid_vals = s["bid_price_med"].to_list()
        ask_vals = s["ask_price_med"].to_list()
        mid_vals = s["mid_price_med"].to_list()
        sanos_vals = s["sanos_price_med"].to_list()

        # Bid-ask shaded band
        fig.add_trace(
            go.Scatter(
                x=ks + ks[::-1],
                y=ask_vals + bid_vals[::-1],
                fill="toself",
                fillcolor=fill_color,
                line=dict(width=0),
                name="bid-ask",
                legendgroup="bid-ask",
                showlegend=show and col_idx == 0,
                hoverinfo="skip",
            ),
            row=row,
            col=col,
        )

        # SANOS fitted line
        fig.add_trace(
            go.Scatter(
                x=ks,
                y=sanos_vals,
                mode="lines+markers",
                name="SANOS fit",
                legendgroup="sanos",
                line=dict(color=color, width=2.5),
                marker=dict(size=6),
                showlegend=show and col_idx == 0,
            ),
            row=row,
            col=col,
        )

        # Market mid crosses
        fig.add_trace(
            go.Scatter(
                x=ks,
                y=mid_vals,
                mode="markers",
                name="market mid",
                legendgroup="mid",
                marker=dict(color=color, symbol="x", size=9, line=dict(width=2)),
                showlegend=show and col_idx == 0,
            ),
            row=row,
            col=col,
        )

fig.update_xaxes(title_text="log-moneyness  (0 = ATM)")
fig.update_yaxes(title_text="price  ($)")
fig.update_layout(
    height=max(380 * n_exp, 420),
    title_text="SANOS fitted prices — calls (left) · puts (right)  (median across day)",
    legend=dict(orientation="h", y=-0.04),
    hovermode="x unified",
)
fig.show()

## Dense interpolation — calls and puts on a fine strike grid

Re-fit one expiry on the OTM quotes and evaluate on a continuous grid of 300 log-moneyness values to visualize the smooth interpolated price curve.

In [ ]:
# Pick expiry with most total OTM quotes
T_r_expr = (pl.col("T") * 200).round(0) / 200  # 0.005 yr bins — one per expiry
df_otm = df_quotes.filter(
    (pl.col("is_call") & (pl.col("log_moneyness") >= 0))
    | (~pl.col("is_call") & (pl.col("log_moneyness") < 0))
)
best_row = (
    df_otm.with_columns(T_r_expr.alias("_T_r"))
    .group_by("_T_r")
    .agg(pl.col("T").first(), pl.len().alias("n"))
    .sort("n", descending=True)
    .head(1)
)
T_best = float(best_row["T"][0])
T_r = float(best_row["_T_r"][0])

# Median forward and discount factor for unnormalizing
sub_q = df_quotes.filter(T_r_expr == T_r)
fwd_med = float(sub_q["forward"].median())
df_med = float(sub_q["discount_factor"].median())
scale = fwd_med * df_med
print(
    f"Expiry T={T_best:.4f} yr  |  forward={fwd_med:.3f}  |  df={df_med:.5f}  |  scale={scale:.3f}"
)

# ── Calls ─────────────────────────────────────────────────────────────────────
grp_c = df_otm.filter((T_r_expr == T_r) & pl.col("is_call")).sort("log_moneyness")
mks_c = grp_c["log_moneyness"].to_numpy().astype(float)
bids_c = (grp_c["bid_price"] / scale).to_numpy().astype(float)
asks_c = (grp_c["ask_price"] / scale).to_numpy().astype(float)
mid_c = (grp_c["observed_price"] / scale).to_numpy().astype(float)
ok_c = np.isfinite(mks_c) & np.isfinite(bids_c) & np.isfinite(asks_c) & (asks_c > bids_c)

anchor_ks_c, q_c, V_atm_c = sanos_option.fit_sanos_call_slice(
    mks_c[ok_c], bids_c[ok_c], asks_c[ok_c], T_best, cfg
)
print(f"Calls: status={'OK' if q_c is not None else 'FAILED'}  V_atm={V_atm_c:.5f}  n={ok_c.sum()}")

# ── Puts ──────────────────────────────────────────────────────────────────────
grp_p = df_otm.filter((T_r_expr == T_r) & ~pl.col("is_call")).sort("log_moneyness")
mks_p = grp_p["log_moneyness"].to_numpy().astype(float)
bids_p = (grp_p["bid_price"] / scale).to_numpy().astype(float)
asks_p = (grp_p["ask_price"] / scale).to_numpy().astype(float)
mid_p = (grp_p["observed_price"] / scale).to_numpy().astype(float)
ok_p = np.isfinite(mks_p) & np.isfinite(bids_p) & np.isfinite(asks_p) & (asks_p > bids_p)

anchor_ks_p, q_p, V_atm_p = sanos_option.fit_sanos_put_slice(
    mks_p[ok_p], bids_p[ok_p], asks_p[ok_p], T_best, cfg
)
print(f"Puts:  status={'OK' if q_p is not None else 'FAILED'}  V_atm={V_atm_p:.5f}  n={ok_p.sum()}")

# ── Dense evaluation grids ────────────────────────────────────────────────────
k_grid_c = np.linspace(max(cfg.k_min, mks_c[ok_c].min() - 0.01), cfg.k_max + 0.01, 300)
k_grid_p = np.linspace(cfg.k_min - 0.01, min(cfg.k_max, mks_p[ok_p].max() + 0.01), 300)

call_dense = (
    sanos_option.eval_sanos_call_prices(q_c, anchor_ks_c, k_grid_c, V_atm_c, cfg.eta) * scale
    if q_c is not None
    else None
)
put_dense = (
    sanos_option.eval_sanos_put_prices(q_p, anchor_ks_p, k_grid_p, V_atm_p, cfg.eta) * scale
    if q_p is not None
    else None
)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig = psp.make_subplots(rows=1, cols=2, subplot_titles=["Calls", "Puts"])

for col_idx, (
    is_call,
    label,
    color,
    fill_color,
    mks,
    bids_raw,
    asks_raw,
    mid_raw,
    ok,
    k_grid,
    dense,
) in enumerate(
    [
        (
            True,
            "Call",
            "royalblue",
            "rgba(65,105,225,0.18)",
            mks_c,
            bids_c,
            asks_c,
            mid_c,
            ok_c,
            k_grid_c,
            call_dense,
        ),
        (
            False,
            "Put",
            "firebrick",
            "rgba(178,34,34,0.18)",
            mks_p,
            bids_p,
            asks_p,
            mid_p,
            ok_p,
            k_grid_p,
            put_dense,
        ),
    ]
):
    col = col_idx + 1
    show = col_idx == 0

    # Bid-ask band at market strikes (dollar prices)
    bids_d = bids_raw[ok] * scale
    asks_d = asks_raw[ok] * scale
    mids_d = mid_raw[ok] * scale
    ks_ok = mks[ok]

    fig.add_trace(
        go.Scatter(
            x=ks_ok.tolist() + ks_ok.tolist()[::-1],
            y=asks_d.tolist() + bids_d.tolist()[::-1],
            fill="toself",
            fillcolor=fill_color,
            line=dict(width=0),
            name="bid-ask",
            legendgroup="ba",
            showlegend=show,
            hoverinfo="skip",
        ),
        row=1,
        col=col,
    )

    # Market mid markers
    fig.add_trace(
        go.Scatter(
            x=ks_ok.tolist(),
            y=mids_d.tolist(),
            mode="markers",
            name="market mid",
            legendgroup="mid",
            marker=dict(color=color, symbol="x", size=10, line=dict(width=2)),
            showlegend=show,
        ),
        row=1,
        col=col,
    )

    # Dense SANOS curve
    if dense is not None:
        fig.add_trace(
            go.Scatter(
                x=k_grid.tolist(),
                y=dense.tolist(),
                mode="lines",
                name="SANOS (dense)",
                legendgroup="sanos",
                line=dict(color=color, width=2.5),
                showlegend=show,
            ),
            row=1,
            col=col,
        )

fig.update_xaxes(title_text="log-moneyness  (0 = ATM)")
fig.update_yaxes(title_text="price  ($)")
fig.update_layout(
    height=460,
    title_text=f"SANOS dense interpolation  T={T_best:.4f} yr  (forward={fwd_med:.2f})",
    legend=dict(orientation="h", y=-0.08),
    hovermode="x",
)
fig.show()

## Risk-neutral density — calls and puts

Both $q_c$ and $q_p$ are probability vectors over anchor strikes.  
The call density's martingale ($\sum q_j e^{k_j} = 1$) checks that the model is arbitrage-free.

In [ ]:
# Uses q_c, q_p, anchor_ks_c, anchor_ks_p computed in the dense-interpolation cell above
bar_width_c = (cfg.k_max - cfg.k_min) / len(anchor_ks_c) * 0.8
bar_width_p = (cfg.k_max - cfg.k_min) / len(anchor_ks_p) * 0.8

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, q, anchor_ks, bar_w, label, color in [
    (axes[0], q_c, anchor_ks_c, bar_width_c, "Call q", "steelblue"),
    (axes[1], q_p, anchor_ks_p, bar_width_p, "Put q", "firebrick"),
]:
    if q is None:
        ax.text(0.5, 0.5, "LP failed", transform=ax.transAxes, ha="center")
        continue
    mart = float(np.exp(anchor_ks) @ q)
    ax.bar(anchor_ks, q, width=bar_w, color=color, alpha=0.8)
    ax.axvline(0, color="black", lw=1, ls="--", label="ATM (k=0)")
    ax.set_xlabel("log-moneyness (anchor)")
    ax.set_ylabel("probability weight q")
    ax.set_title(f"SANOS {label}  (T={T_best:.4f})  mart={mart:.5f}  Σq={q.sum():.5f}")
    ax.legend()

plt.tight_layout()
plt.show()

## Arbitrage check — model prices vs bid/ask bounds

For each market quote, check whether the SANOS model price lies within the bid/ask spread.

In [ ]:
# For the fitted slice: model prices vs bid/ask at market strikes
fig = psp.make_subplots(rows=1, cols=2, subplot_titles=["Calls", "Puts"], shared_yaxes=False)

for col_idx, (is_call, label, color, q, anchor_ks, V_atm, mks, bids_raw, asks_raw, ok) in enumerate(
    [
        (True, "call", "royalblue", q_c, anchor_ks_c, V_atm_c, mks_c, bids_c, asks_c, ok_c),
        (False, "put", "firebrick", q_p, anchor_ks_p, V_atm_p, mks_p, bids_p, asks_p, ok_p),
    ]
):
    col = col_idx + 1
    if q is None:
        continue

    ks_ok = mks[ok]
    bids_d = bids_raw[ok] * scale
    asks_d = asks_raw[ok] * scale

    # Model prices at market strikes (dollar)
    if is_call:
        model_norm = sanos_option.eval_sanos_call_prices(q, anchor_ks, ks_ok, V_atm, cfg.eta)
    else:
        model_norm = sanos_option.eval_sanos_put_prices(q, anchor_ks, ks_ok, V_atm, cfg.eta)
    model_d = model_norm * scale

    inside = ((model_d >= bids_d) & (model_d <= asks_d)).mean()
    print(f"{label}: {inside:.1%} of quotes inside bid-ask")

    fig.add_trace(
        go.Scatter(
            x=ks_ok.tolist(),
            y=bids_d.tolist(),
            mode="markers",
            name="bid",
            marker=dict(color="green", symbol="triangle-up", size=8),
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=ks_ok.tolist(),
            y=asks_d.tolist(),
            mode="markers",
            name="ask",
            marker=dict(color="red", symbol="triangle-down", size=8),
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=ks_ok.tolist(),
            y=model_d.tolist(),
            mode="markers",
            name="SANOS model",
            marker=dict(color=color, symbol="circle", size=7),
            showlegend=(col == 1),
        ),
        row=1,
        col=col,
    )

fig.update_xaxes(title_text="log-moneyness")
fig.update_yaxes(title_text="price  ($)")
fig.update_layout(
    height=440,
    title_text=f"Model prices vs bid-ask  T={T_best:.4f} yr",
    legend=dict(orientation="h", y=-0.1),
)
fig.show()